For this experimentation run we will freeze all layers except for the last three encoding layers, so in essence we will train four layers, the classification layer with the largest learning rate of 0.2, and the subsequently we will decrease the learning rate by a factor of 0.2. 

further areas of exploration:
- Use a lr decay such as a cosine schedular
- Increasing the trainable layers

In [1]:
import torchvision
import torch
from torchvision import datasets, transforms
from torchvision.transforms import v2
import torch.nn as nn
import torchinfo
import wandb
from tqdm import tqdm
from pathlib import Path 
import os
import time
import gc
from torch.utils.data import DataLoader

In [2]:
lr = 1e-2 
lr_decay = 0.9
EPOCHS = 75
vit_data_dir=Path("../data")
NUM_WORKERS = 0 
BATCH_SIZE = 32

vit_weights = torchvision.models.ViT_B_32_Weights.DEFAULT
vit_transforms = vit_weights.transforms()

vit_train_transforms = transforms.Compose([
    transforms.TrivialAugmentWide(),
    vit_transforms
])

In [3]:
vit_transforms = vit_weights.transforms()

transform = v2.Compose([
    vit_transforms,
    v2.ToTensor()
])
train_data = datasets.ImageFolder(root="../data/train", transform=transform, target_transform=None)
test_data = datasets.ImageFolder(root="../data/test", transform=transform, target_transform=None)

train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

/opt/homebrew/Caskroom/miniconda/base/envs/AircraftDetection/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [4]:
# Properly set device with error handling
if torch.backends.mps.is_available():
    device = "mps"
    print("[INFO] Using MPS (Metal Performance Shaders)")
elif torch.cuda.is_available():
    device = "cuda"
    print("[INFO] Using CUDA")
else:
    device = "cpu"
    print("[INFO] Using CPU")

print(f"[INFO] Device set to: {device}")

[INFO] Using MPS (Metal Performance Shaders)
[INFO] Device set to: mps


In [5]:
def create_vit(device,
               lr, 
               lr_decay,
               num_classes: int = 100,
               seed: int = 42):
    # Set random seeds properly based on device
    torch.manual_seed(seed)
    if device == "mps" and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    elif device == "cuda" and torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    weights = torchvision.models.ViT_B_32_Weights.DEFAULT
    model = torchvision.models.vit_b_32(weights=weights)

    # Replace the head with the correct number of classes
    model.heads = nn.Sequential(
        nn.Linear(in_features=768, out_features=num_classes)
    )

    layer_names = {
        "encoder_layer_9": [],
        "encoder_layer_10": [],
        "encoder_layer_11": [],
        "heads": []
    }

    # Set requires_grad to False for all parameters first
    for _, (name, param) in enumerate(model.named_parameters()):
        param.requires_grad = False
        for key in layer_names:
            if key in name:
                layer_names[key].append(param) 
                param.requires_grad = True

    parameters = []

    # Create parameter groups with decreasing learning rates
    for key in layer_names:
        if layer_names[key]:  # Only add if there are parameters
            parameters += [{'params': [p for p in layer_names[key]], 'lr': lr}]
            lr = lr * lr_decay

    # Move model to device after setup
    model = model.to(device)
    
    return model, parameters

In [6]:
model, parameters = create_vit(device=device, lr=lr, lr_decay=lr_decay)
print(f"[INFO] Model created and moved to {device}")
print(f"[INFO] Number of parameter groups: {len(parameters)}")
for i, param_group in enumerate(parameters):
    print(f"[INFO] Parameter group {i}: lr={param_group['lr']}, params={len(param_group['params'])}")

[INFO] Model created and moved to mps
[INFO] Number of parameter groups: 4
[INFO] Parameter group 0: lr=0.01, params=12
[INFO] Parameter group 1: lr=0.009000000000000001, params=12
[INFO] Parameter group 2: lr=0.008100000000000001, params=12
[INFO] Parameter group 3: lr=0.007290000000000001, params=2


In [7]:
torchinfo.summary(model=model, 
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
VisionTransformer (VisionTransformer)                        [32, 3, 224, 224]    [32, 100]            768                  Partial
├─Conv2d (conv_proj)                                         [32, 3, 224, 224]    [32, 768, 7, 7]      (2,360,064)          False
├─Encoder (encoder)                                          [32, 50, 768]        [32, 50, 768]        38,400               Partial
│    └─Dropout (dropout)                                     [32, 50, 768]        [32, 50, 768]        --                   --
│    └─Sequential (layers)                                   [32, 50, 768]        [32, 50, 768]        --                   Partial
│    │    └─EncoderBlock (encoder_layer_0)                   [32, 50, 768]        [32, 50, 768]        (7,087,872)          False
│    │    └─EncoderBlock (encoder_layer_1)                   [32, 50, 768]        [

In [8]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader:  torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          device,
          epochs: int = 10,
          save_model: bool = False,
          save_model_path: str = "./models",
          model_name: str = "vit_model"):
    
    if device == "mps":
        torch.mps.empty_cache()
        gc.collect()
        print("[INFO] MPS cache cleared and garbage collected")
    
    print("[INFO] Initializing wandb...")
    wandb.init(project="AircraftDetection", config={"epochs": epochs, "model name":model_name})

    log_interval = 10

    for epoch in tqdm(range(epochs)):
        model.to(device)
        model.train()

        train_loss, train_acc = 0, 0
        running_loss, running_acc = 0, 0
        start_time = time.time()

        for batch, (X, y) in enumerate(train_dataloader):
            X, y = X.to(device), y.to(device)

            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            train_loss += loss.item()
            running_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            acc = (y_pred_class == y).sum().item()/len(y_pred)
            train_acc += acc
            running_acc += acc

            if device == "mps" and batch % 5 == 0:
                torch.mps.empty_cache()
                if batch % 20 == 0:
                    gc.collect()

                if (batch + 1) % log_interval == 0 or (batch + 1) == len(train_dataloader):
                    elapsed = time.time() - start_time
                    avg_loss = running_loss / log_interval if (batch + 1) % log_interval == 0 else running_loss / ((batch + 1) % log_interval)
                    avg_acc = running_acc / log_interval if (batch + 1) % log_interval == 0 else running_acc / ((batch + 1) % log_interval)
                    print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch+1}/{len(train_dataloader)}] "
                          f"Avg Loss: {avg_loss:.4f} Avg Acc: {avg_acc:.4f} "
                          f"Elapsed: {elapsed:.1f}s")
                    running_loss, running_acc = 0, 0
                    start_time = time.time()
                    

        train_loss = train_loss / len(train_dataloader)
        train_acc = train_acc / len(train_dataloader)
        print(f"[INFO] Epoch {epoch+1} training complete. Avg train_loss: {train_loss:.4f}, Avg train_acc: {train_acc:.4f}")

        model.eval()
        test_loss, test_acc = 0, 0
        with torch.inference_mode():
            for batch, (X, y) in enumerate(test_dataloader):
                X, y = X.to(device), y.to(device)
                if batch == 0:
                    print("[INFO] First test batch loaded and moved to device.")

                # Forward pass
                test_pred_logits = model(X)
                if batch == 0:
                    print("[INFO] Forward pass complete for first test batch.")

                # Calculate and accumulate loss
                loss = loss_fn(test_pred_logits, y)
                test_loss += loss.item()
                if batch == 0:
                    print(f"[INFO] Test loss computed for first batch: {loss.item():.4f}")

                # Calculate and accumulate accuracy
                test_pred_labels = test_pred_logits.argmax(dim=1) 
                acc = ((test_pred_labels == y).sum().item() / len(test_pred_labels))
                test_acc += acc
                if batch == 0:
                    print(f"[INFO] Test accuracy computed for first batch: {acc:.4f}")
                    print(f"[INFO] First test batch - Predicted classes: {test_pred_labels[:5]}")
                    print(f"[INFO] First test batch - True labels: {y[:5]}")
                    print(f"[INFO] First test batch - Prediction probabilities (max): {torch.softmax(test_pred_logits, dim=1).max(dim=1)[0][:5]}")

            test_loss = test_loss / len(test_dataloader)
            test_acc = test_acc / len(test_dataloader)
            print(f"[INFO] Evaluation complete. Avg test_loss: {test_loss:.4f}, Avg test_acc: {test_acc:.4f}")
            
            results = {"train_loss": [],
                        "train_acc": [],
                        "test_loss": [],
                        "test_acc": []}
            print(
                    f"Epoch: {epoch+1} | "
                    f"train_loss: {train_loss:.4f} | "
                    f"train_acc: {train_acc:.4f} | "
                    f"test_loss: {test_loss:.4f} | "
                    f"test_acc: {test_acc:.4f}")
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            print("[INFO] Logging metrics to wandb.")
            wandb.log({"test_loss": test_loss, "test_acc": test_acc, "train_loss": train_loss, "train_acc": train_acc})

    if save_model == True:
        print(f"[INFO] Saving {model_name} model to {save_model_path}")
        MODEL_PATH = Path(save_model_path) 
        MODEL_PATH.mkdir(parents=True,
                         exist_ok=True)
        MODEL_SAVE_PATH = MODEL_PATH/model_name
        torch.save(obj=model.state_dict(),
                   f=MODEL_SAVE_PATH)
        print(f"[INFO] Model saved to {MODEL_SAVE_PATH}")
        return results
    else:
        print("[INFO] Training complete. Model not saved.")
        return results

In [9]:
vit_loss_fn = torch.nn.CrossEntropyLoss()
vit_optimizer = torch.optim.Adam(params=parameters)

train(model=model,
      train_dataloader=train_dataloader,
      test_dataloader=train_dataloader,
      loss_fn=vit_loss_fn,
      optimizer=vit_optimizer,
      device=device,
      epochs=EPOCHS,
      save_model=True,
      save_model_path="./models",
      model_name=f"layer_wise_learning_rate")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


[INFO] MPS cache cleared and garbage collected
[INFO] Initializing wandb...


wandb: Currently logged in as: pypdeveloper. Use `wandb login --relogin` to force relogin


  0%|          | 0/75 [00:00<?, ?it/s]

Epoch [1/75] Batch [1251/1251] Avg Loss: 4877.8289 Avg Acc: 85.4062 Elapsed: 587.8s
[INFO] Epoch 1 training complete. Avg train_loss: 3.8991, Avg train_acc: 0.0683
[INFO] First test batch loaded and moved to device.
[INFO] Forward pass complete for first test batch.
[INFO] Test loss computed for first batch: 3.6770
[INFO] Test accuracy computed for first batch: 0.1250
[INFO] First test batch - Predicted classes: tensor([ 6, 63, 36, 36,  6], device='mps:0')
[INFO] First test batch - True labels: tensor([ 7, 60, 36, 42, 27], device='mps:0')
[INFO] First test batch - Prediction probabilities (max): tensor([0.2239, 0.2206, 0.3142, 0.1732, 0.0969], device='mps:0')


...

[INFO] Evaluation complete. Avg test_loss: 0.0427, Avg test_acc: 0.9860
Epoch: 74 | train_loss: 0.0630 | train_acc: 0.9803 | test_loss: 0.0427 | test_acc: 0.9860
[INFO] Logging metrics to wandb.
Epoch [75/75] Batch [1251/1251] Avg Loss: 74.6694 Avg Acc: 1227.4062 Elapsed: 630.6s
[INFO] Epoch 75 training complete. Avg train_loss: 0.0597, Avg train_acc: 0.9811
[INFO] First test batch loaded and moved to device.
[INFO] Forward pass complete for first test batch.
[INFO] Test loss computed for first batch: 0.1269
[INFO] Test accuracy computed for first batch: 0.9375
[INFO] First test batch - Predicted classes: tensor([85, 57, 52, 15, 12], device='mps:0')
[INFO] First test batch - True labels: tensor([85, 57, 52, 15, 12], device='mps:0')
[INFO] First test batch - Prediction probabilities (max): tensor([0.9999, 1.0000, 1.0000, 0.9926, 0.9983], device='mps:0')


100%|██████████| 75/75 [24:17:11<00:00, 1165.75s/it]

[INFO] Evaluation complete. Avg test_loss: 0.0871, Avg test_acc: 0.9727
Epoch: 75 | train_loss: 0.0597 | train_acc: 0.9811 | test_loss: 0.0871 | test_acc: 0.9727
[INFO] Logging metrics to wandb.
[INFO] Saving layer_wise_learning_rate model to ./models


[INFO] Model saved to models/layer_wise_learning_rate


{'train_loss': [0.05968777526304344],
 'train_acc': [0.9811400879296562],
 'test_loss': [0.0870958184604359],
 'test_acc': [0.9727218225419664]}